# VAE output visualization

This notebook loads the VAE checkpoint produced by `src/volumetric_jepa_vae_submission.py` and visualizes:
- dataset3 T2 original slices vs VAE reconstructions
- dataset1 paired T2 slices vs pseudo dataset3-like T2 reconstructions

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch

REPO_ROOT = Path.cwd()
if (REPO_ROOT / 'src').exists():
    sys.path.insert(0, str(REPO_ROOT / 'src'))
else:
    sys.path.insert(0, str(REPO_ROOT.parent / 'src'))

from volumetric_jepa_vae_submission import ConvVAE, resolve_path, volume_to_slice_stack

In [ ]:
DATA_ROOT = Path('data/ehl-paris-medical-image-retrieval')

# Kaggle fallback paths. Override these two variables if your dataset or artifacts are elsewhere.
if not DATA_ROOT.exists():
    DATA_ROOT = Path('/kaggle/input/competitions/ehl-paris-medical-image-retrieval')

# Search for the VAE checkpoint in multiple possible locations:
# 1. artifacts/vae_jepa/vae.pt (default local)
# 2. artifacts/vae_jepa_smoke/vae.pt (smoke test local)
# 3. /tmp/hack_paris/artifacts/vae_jepa/vae.pt (remote Jupyter GPU)
# 4. /kaggle/working/artifacts/vae_jepa/vae.pt (default Kaggle)
# 5. /kaggle/working/artifacts/vae_jepa_smoke/vae.pt (smoke test Kaggle)
VAE_CKPT = None
for candidate in [
    Path('artifacts/vae_jepa/vae.pt'),
    Path('artifacts/vae_jepa_smoke/vae.pt'),
    Path('/tmp/hack_paris/artifacts/vae_jepa/vae.pt'),
    Path('/kaggle/working/artifacts/vae_jepa/vae.pt'),
    Path('/kaggle/working/artifacts/vae_jepa_smoke/vae.pt'),
]:
    if candidate.exists():
        VAE_CKPT = candidate
        break

if VAE_CKPT is None:
    # Default to the standard path if none exists so the error trace shows what is expected
    VAE_CKPT = Path('artifacts/vae_jepa/vae.pt')

CACHE_PT = None
for candidate in [
    Path('data/mri_volumetric_cache_step5_96_uint8.pt'),
    Path('/tmp/hack_paris/data/mri_volumetric_cache_step5_96_uint8.pt'),
]:
    if candidate.exists():
        CACHE_PT = candidate
        break

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DATA_ROOT =', DATA_ROOT)
print('VAE_CKPT =', VAE_CKPT)
print('CACHE_PT =', CACHE_PT)
print('device =', device)

In [ ]:
if not VAE_CKPT.exists():
    raise FileNotFoundError(
        f'Missing {VAE_CKPT}. Run scripts/run_vae_jepa_submission.sh first, '
        'or set VAE_CKPT to an existing checkpoint.'
    )

ckpt = torch.load(VAE_CKPT, map_location=device, weights_only=False)
vae = ConvVAE(latent_dim=ckpt['latent_dim']).to(device)
vae.load_state_dict(ckpt['state_dict'])
vae.eval()
image_size = int(ckpt['image_size'])
cache = torch.load(CACHE_PT, map_location='cpu', weights_only=False) if CACHE_PT is not None else None
print('loaded VAE:', {'latent_dim': ckpt['latent_dim'], 'image_size': image_size})

In [ ]:
def middle_valid_slice(stack, mask):
    valid = torch.nonzero(mask, as_tuple=False).flatten()
    return int(valid[len(valid) // 2])

def payload_tensor(payload):
    tensor = torch.as_tensor(payload['tensor'])
    if tensor.dtype == torch.uint8:
        tensor = tensor.float() / 255.0
    return tensor.float()

@torch.no_grad()
def reconstruct_slice(path, slice_step=5, max_slices=32):
    stack, mask = volume_to_slice_stack(Path(path), image_size=image_size, slice_step=slice_step, max_slices=max_slices)
    idx = middle_valid_slice(stack, mask)
    original = stack[idx:idx + 1].to(device)
    reconstruction, mu, logvar = vae(original)
    return original.cpu().squeeze().numpy(), reconstruction.cpu().squeeze().numpy(), idx

@torch.no_grad()
def reconstruct_payload(payload):
    stack = payload_tensor(payload)
    mask = torch.as_tensor(payload['mask']).bool()
    idx = middle_valid_slice(stack, mask)
    original = stack[idx:idx + 1].to(device)
    reconstruction, mu, logvar = vae(original)
    return original.cpu().squeeze().numpy(), reconstruction.cpu().squeeze().numpy(), idx

def first_cached_dataset3_t2():
    for payload in cache['images'].values():
        if payload.get('dataset') == 'dataset3' and 'T2' in str(payload.get('modality', '')):
            return payload
    raise ValueError('No dataset3 T2 payload found in cache')

def first_cached_dataset1_pair_target():
    first_pair = cache['train_pairs'][0]
    return cache['images'][first_pair['target_id']]

In [ ]:
if cache is not None:
    d3_payload = first_cached_dataset3_t2()
    d1_payload = first_cached_dataset1_pair_target()
    d3_original, d3_recon, d3_idx = reconstruct_payload(d3_payload)
    d1_original, d1_pseudo, d1_idx = reconstruct_payload(d1_payload)
    d3_name = d3_payload['id']
    d1_name = d1_payload['id']
else:
    dataset3_gallery = pd.read_csv(DATA_ROOT / 'dataset3/val_gallery.csv')
    dataset1_pairs = pd.read_csv(DATA_ROOT / 'dataset1/train_pairs.csv')
    dataset3_t2_path = resolve_path(DATA_ROOT, dataset3_gallery.iloc[0]['target_image'])
    dataset1_t2_path = resolve_path(DATA_ROOT, dataset1_pairs.iloc[0]['target_image'])
    d3_original, d3_recon, d3_idx = reconstruct_slice(dataset3_t2_path)
    d1_original, d1_pseudo, d1_idx = reconstruct_slice(dataset1_t2_path)
    d3_name = dataset3_gallery.iloc[0]['target_id']
    d1_name = dataset1_pairs.iloc[0]['target_id']

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
items = [
    (d3_original, f'{d3_name} dataset3 T2 original - slice {d3_idx}'),
    (d3_recon, 'dataset3 T2 VAE reconstruction'),
    (d1_original, f'{d1_name} dataset1 T2 original - slice {d1_idx}'),
    (d1_pseudo, 'dataset1 pseudo dataset3-like T2'),
]
for ax, (image, title) in zip(axes.ravel(), items):
    ax.imshow(image, cmap='gray')
    ax.set_title(title, fontsize=10)
    ax.axis('off')
plt.tight_layout()

In [ ]:
# Visualize several pseudo-T2 outputs from dataset1.
if cache is not None:
    cached_pairs = [cache['images'][row['target_id']] for row in cache['train_pairs']]
    n = min(6, len(cached_pairs))
else:
    dataset1_pairs = pd.read_csv(DATA_ROOT / 'dataset1/train_pairs.csv')
    n = min(6, len(dataset1_pairs))
fig, axes = plt.subplots(n, 2, figsize=(7, 3 * n))
if n == 1:
    axes = axes.reshape(1, 2)

for row_idx in range(n):
    if cache is not None:
        payload = cached_pairs[row_idx]
        original, pseudo, idx = reconstruct_payload(payload)
        target_id = payload['id']
    else:
        row = dataset1_pairs.iloc[row_idx]
        path = resolve_path(DATA_ROOT, row['target_image'])
        original, pseudo, idx = reconstruct_slice(path)
        target_id = row['target_id']
    axes[row_idx, 0].imshow(original, cmap='gray')
    axes[row_idx, 0].set_title(f"{target_id} original slice {idx}", fontsize=9)
    axes[row_idx, 0].axis('off')
    axes[row_idx, 1].imshow(pseudo, cmap='gray')
    axes[row_idx, 1].set_title(f"{target_id} pseudo T2", fontsize=9)
    axes[row_idx, 1].axis('off')

plt.tight_layout()